# Scenarios and weights

Base, Upside and Downside paths are internally generated assumptions. IFRS 9 does not prescribe their names or probabilities.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

In [2]:
weights = query('select * from scenario_weight_analysis'); weights

,scenario,configured_weight,historical_analogue_frequency,configured_component,historical_component,selected_weight,weighting_method
0,Base,0.6000,0.5596,judgemental starting probability,nearest historical macro-regime frequency,0.5798,blended_historical_analogue
1,Downside,0.2000,0.1387,judgemental starting probability,nearest historical macro-regime frequency,0.1693,blended_historical_analogue
2,Upside,0.2000,0.3017,judgemental starting probability,nearest historical macro-regime frequency,0.2509,blended_historical_analogue


In [3]:
query('''select scenario,
avg(case when month_number<=12 then unemployment_rate end) unemployment_first_year,
avg(case when month_number<=12 then gdp_growth_yoy end) gdp_first_year,
avg(case when month_number<=12 then hpi_growth_yoy end) hpi_first_year
from macro_scenarios group by scenario''')

,scenario,unemployment_first_year,gdp_first_year,hpi_first_year
0,Base,4.2677,2.6347,2.5520
1,Downside,6.7677,-0.3653,-5.4480
2,Upside,3.5177,3.6347,4.5520


Production weights blend the configured judgement and historical analogue frequency in equal proportions. The result is a modelling choice, not an official forecast or an IFRS 9 requirement.

In [4]:
query('select * from scenario_summary order by scenario')

,scenario,scenario_weight,scenario_ecl,weighted_contribution
0,Base,0.5798,"419,658.5693","243,320.2848"
1,Downside,0.1693,"458,859.3606","77,704.6508"
2,Upside,0.2509,"410,509.3905","102,976.9298"
